### 1. 📥​ Imports

In [12]:
import sys, os
from pathlib import Path
import pandas as pd

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

from tcibench.preparation import (Dataset, DatasetMetadata, load_dataset, load_metadata)

from tcibench.generation.llms import (
    LLMClient, LLMConfig,
    load_sampling_plan, load_schema,
    run_generation,
    get_context, build_generation_messages
)

### 2. ⚙️​ Configs

In [13]:
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "tcibench" / "preparation" / "metadata.json").is_file()),
    None,
)

In [14]:
# Reproducibility
SPLIT_SEED = 42
SAMPLING_SEED = 42
GENERATION_SEED = 42

# Generation protocol
EXAMPLES_PER_CLASS = 10
RECORDS_PER_BATCH = 10

📂​ Datasets

In [ ]:
DATASETS = {
    "diabetes": "Diabetes",
    "sick": "Thyroid Sick",
    "predictive_maintenance": "Predictive Maintenance",
    "adult": "Adult Income"
}

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / f"split_seed_{SPLIT_SEED}"

RAW_PATHS = {
    "diabetes": RAW_ROOT / "diabetes.csv",
    "sick": RAW_ROOT / "sick.arff",
    "predictive_maintenance": RAW_ROOT / "predictive_maintenance.csv",
    "adult": RAW_ROOT / "adult.csv"
}

🤖 LLMs

In [16]:
# API keys
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

# LLMs
MODELS = {
    "deepseek_v4_flash": LLMConfig(
        model="deepseek-ai/deepseek-v4-flash-0731",
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY,
        max_tokens=16384,
        temperature=0.7,
        generation_kwargs={
            "top_p": 0.95,
            "extra_body": {
                "chat_template_kwargs": {
                    "thinking": False,
                    "reasoning_effort": "high",
                }
            },
        },
    ),

    "phi_4_mini_instruct": LLMConfig(
        model="phi-4-mini-instruct",
        base_url="http://192.168.68.124:1234/v1",
        api_key="lm-studio",
        max_tokens=4096,
        temperature=0.7,
    ),

    "deepseek_r1_distill_llama_8b": LLMConfig(
        model="deepseek-r1-distill-llama-8b",
        base_url="http://192.168.68.124:1234/v1",
        api_key="lm-studio",
        max_tokens=4096,
        temperature=0.7,
    ),

    "qwen3_4b": LLMConfig(
        model="qwen3-4b",
        base_url="http://192.168.68.124:1234/v1",
        api_key="lm-studio",
        max_tokens=4096,
        temperature=0.7,
    ),

    "meta_llama_3_1_8b_instruct": LLMConfig(
        model="meta-llama-3.1-8b-instruct",
        base_url="http://192.168.68.124:1234/v1",
        api_key="lm-studio",
        max_tokens=4096,
        temperature=0.7,
    ),

    "qwen3_1_7b": LLMConfig(
        model="qwen3-1.7b:3",
        base_url="http://192.168.68.124:1234/v1",
        api_key="lm-studio",
        max_tokens=4096,
        temperature=0.7,
    ),

    "kimi_k3": LLMConfig(
        model="moonshotai/kimi-k3",
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY,
        max_tokens=16384,
        temperature=1.0,
        generation_kwargs={
            "reasoning_effort": "max",
        },
    ),
    "glm_5_3": LLMConfig(
        model="z-ai/glm-5.3",
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY,
        max_tokens=4096,
        temperature=1.0,
        generation_kwargs={
            "stream": False,
            "reasoning_effort": "low",
        },
    ),
    "gemma_4_31b": LLMConfig(
        model="google/gemma-4-31b-it",
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY,
        max_tokens=4096,
        temperature=1.0,
        generation_kwargs={
            "extra_body": {
                "chat_template_kwargs": {
                    "enable_thinking": False,
                },
            },
            "stream": False,
        },
        ),
}

### 3. 📂​ Dataset

In [17]:
def inspect_dataset(
    data: pd.DataFrame,
    metadata: DatasetMetadata,
    cleaned: pd.DataFrame,
    cleaning_stats: dict,
):
    """Inspect a dataset before running the experiment."""

    print(f"Dataset: {metadata.display_name}")
    print(f"Target: {metadata.target_column}")

    print("\nRaw data")
    print(f"Shape: {data.shape}")
    display(data.head())

    print("\nCleaning")
    print(f"Shape after cleaning: {cleaned.shape}")
    print(f"Removed columns: {cleaning_stats['dropped_columns']}")
    print(f"Removed rows with missing values: {cleaning_stats['removed_missing_rows']}")
    print(f"Removed duplicate rows: {cleaning_stats['removed_duplicate_rows']}")

    print("\nCleaned data")
    display(cleaned.head())

    print("\nTarget distribution")
    target = metadata.target_column
    display(pd.DataFrame({
        "count": cleaned[target].value_counts(),
        "proportion": cleaned[target].value_counts(normalize=True).mul(100).round(2),
    }))

    target_counts = cleaned[target].value_counts()
    print(f"Imbalance ratio: {target_counts.max() / target_counts.min():.2f}")

In [ ]:
DATASET_ID = "adult"

metadata = load_metadata(DATASET_ID)
data = load_dataset(RAW_PATHS[DATASET_ID])

dataset = Dataset(data, metadata)

cleaned, cleaning_report = dataset.preprocessing.clean(data)
folds = dataset.preprocessing.split(cleaned)

inspect_dataset(data, metadata, cleaned, cleaning_report)

train, test, audit = folds[0]

counts = train[metadata.target_column].value_counts()
majority_class = counts.idxmax()
minority_class = counts.idxmin()

majority_count = int(counts[majority_class])
minority_count = int(counts[minority_class])
records_to_generate = majority_count - minority_count
imbalance_ratio = majority_count / minority_count

print(f"Dataset: {metadata.display_name}")
print(f"Cleaned: {cleaned.shape}")
print(f"Fold 1 — Train: {train.shape} | Test: {test.shape}")
print(f"Majority: {majority_class} ({majority_count})")
print(f"Minority: {minority_class} ({minority_count})")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
print(f"Records to generate: {records_to_generate}")

Dataset: AI4I 2020 Predictive Maintenance
Target: Target

Raw data
Shape: (10000, 10)


,UDI,Product ID,Type,Air temperature (K),Process temperature (K),Rotational speed (rpm),Torque (Nm),Tool wear (min),Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure



Cleaning
Shape after cleaning: (10000, 7)
Removed columns: ['UDI', 'Product ID', 'Failure Type']
Removed rows with missing values: 0
Removed duplicate rows: 0

Cleaned data


,Type,Air temperature (K),Process temperature (K),Rotational speed (rpm),Torque (Nm),Tool wear (min),Target
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0



Target distribution


,count,proportion
Target,,
0,9661,96.61
1,339,3.39


Imbalance ratio: 28.50
Dataset: AI4I 2020 Predictive Maintenance
Cleaned: (10000, 7)
Fold 1 — Train: (8000, 7) | Test: (2000, 7)
Majority: 0 (7728)
Minority: 1 (272)
Imbalance ratio: 28.41
Records to generate: 7456


In [19]:
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"

sampling_plans = []
schemas = []

for fold_id, (train, test, audit) in enumerate(folds):
    counts = train[metadata.target_column].value_counts()
    majority_class = counts.idxmax()
    minority_class = counts.idxmin()

    records_to_generate = int(counts[majority_class] - counts[minority_class])

    sampling_plan = load_sampling_plan(
        train,
        metadata,
        records_to_generate=records_to_generate,
        records_per_batch=RECORDS_PER_BATCH,
        examples_per_class=EXAMPLES_PER_CLASS,
        fold_id=fold_id,
        sampling_seed=SAMPLING_SEED,
        output_root=ARTIFACTS_ROOT / "generation" / "sampling",
    )

    schema = load_schema(
        train,
        metadata,
        minority_class=minority_class,
        fold_id=fold_id,
        records_per_batch=RECORDS_PER_BATCH,
        output_root=ARTIFACTS_ROOT / "generation" / "schemas",
    )

    sampling_plans.append(sampling_plan)
    schemas.append(schema)

### 4. 📄​ Prompt example

In [20]:
# Prompt example from fold 0
fold_id = 0

train, test, audit = folds[fold_id]
sampling_plan = sampling_plans[fold_id]

counts = train[metadata.target_column].value_counts()
minority_class = counts.idxmin()

context = get_context(
    train,
    sampling_plan,
    dataset_id=metadata.dataset_id,
    call_index=0,
)

messages = build_generation_messages(
    context,
    metadata,
    minority_class=minority_class,
    records_per_batch=RECORDS_PER_BATCH,
    examples_per_class=EXAMPLES_PER_CLASS,
)

for message in messages:
    print(f"\n--- {message['role'].upper()} ---\n")
    print(message["content"])


--- SYSTEM ---

You are a tabular synthetic data generator.

Given the dataset context provided, generate new synthetic
samples that are plausible within the dataset domain.

Use the dataset description, column descriptions, and reference examples
to understand the meaning of the attributes and the relationships among them.

Each attribute value must be plsausible on its own and consistent with the others.
Preserve realistic relationships, dependencies, and correlations among
attributes within the context of the dataset domain.

Do not copy, reproduce, or minimally modify any reference record.
Generate genuinely new combinations of attribute values.

--- USER ---

Dataset description:
Synthetic dataset modelled on industrial predictive maintenance: sensor measurements for predicting machine failures.

Target column:
Target

Column descriptions:
Type: Product quality variant: L (low), M (medium) or H (high).
Air temperature (K): Air temperature in kelvin.
Process temperature (K): Proce

### 5. 🤖 LLMs generation

In [21]:
RUN_MODELS = [
    "gemma_4_31b",
]

In [22]:
for fold_id, (train, test, audit) in enumerate(folds):
    schema = schemas[fold_id]
    sampling_plan = sampling_plans[fold_id]

    for model_name in RUN_MODELS:
        config = MODELS[model_name]
        client = LLMClient(config)

        result = run_generation(
            train,
            metadata,
            schema,
            sampling_plan,
            client,
            test=test,
            fold_id=fold_id,
            generation_seed=GENERATION_SEED,
            max_concurrent_calls=8,
            output_root=PROJECT_ROOT / "artifacts" / "generation",
        )

        print(result.summary)


Dataset: predictive_maintenance | Model: google/gemma-4-31b-it | Fold: 0 | Minority class: 1 | Target records: 7456 | Planned contexts: 746

Context 1/746 | Attempt 1 | Returned: 10 | Valid: 10 | Accepted: 10 | Block: 10/10 | Total: 10/7456
Requested class: Target=1
Returned valid groups: {1: 10}
Discarded -> Format/schema: 0 | Duplicate: 0 | Copied prompt: 0 | Copied train/test: 0
Input tokens: 1664 | Output tokens: 675

Context 2/746 | Attempt 1 | Returned: 10 | Valid: 10 | Accepted: 10 | Block: 10/10 | Total: 20/7456
Requested class: Target=1
Returned valid groups: {1: 10}
Discarded -> Format/schema: 0 | Duplicate: 0 | Copied prompt: 0 | Copied train/test: 0
Input tokens: 1664 | Output tokens: 687

Context 3/746 | Attempt 1 | Returned: 10 | Valid: 10 | Accepted: 10 | Block: 10/10 | Total: 30/7456
Requested class: Target=1
Returned valid groups: {1: 10}
Discarded -> Format/schema: 0 | Duplicate: 0 | Copied prompt: 0 | Copied train/test: 0
Input tokens: 1663 | Output tokens: 804

Con